# Inferencja modelu Qwen3.5 do klasyfikacji utworów rapowych


## Import bibliotek


In [ ]:
import os

import numpy as np
import pandas as pd
import torch
from peft import PeftModel

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

# Unsloth przed transformers (optymalizacje i patch Qwen3.5)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

from transformers import AutoModelForSequenceClassification, AutoTokenizer

## Wczytanie danych i etykiet


In [ ]:
DATASET_DIR = "dataset"

# Etykiety odtwarzamy z train.json, a zwrotki do inferencji bierzemy ze zbioru testowego.
train_df = pd.read_json(os.path.join(DATASET_DIR, "train.json"))
test_df = pd.read_json(os.path.join(DATASET_DIR, "test.json"))

labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(labels)

# Zostawiamy tylko zwrotki testowe o etykietach znanych modelowi.
test_df = test_df[test_df["label"].isin(label2id)].reset_index(drop=True)

print(f"Liczba klas: {NUM_LABELS}")
print(f"Zbiór testowy: {len(test_df)} zwrotek")
test_df.head(3)

## Wczytanie modelu bazowego i adapterów LoRA

Model bazowy `unsloth/Qwen3.5-4B` ładujemy w bf16 LoRA na GPU, a na sprzęcie bez GPU na CPU bez kwantyzacji.
Następnie nakładamy zapisane adaptery LoRA i wytrenowaną głowicę klasyfikacyjną z katalogu `qwen35-4b-polish-rap-classifier` przez `PeftModel.from_pretrained`.


In [ ]:
MODEL_NAME = "unsloth/Qwen3.5-4B"
ADAPTER_DIR = "qwen35-4b-polish-rap-classifier"
MAX_LENGTH = 600  # ok. 99. percentyl

has_cuda = torch.cuda.is_available()
print(f"Urządzenie: {'GPU (bf16 LoRA)' if has_cuda else 'CPU (bez kwantyzacji)'}")

# Wczytanie modelu bazowego
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=has_cuda,
    dtype=None if has_cuda else torch.float32,
    auto_model=AutoModelForSequenceClassification,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=False,
    ignore_mismatched_sizes=True,
    use_exact_model_name=True,
    device_map="auto",
)

# Nałożenie wytrenowanych adapterów LoRA i głowicy klasyfikacyjnej.
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

# Tokenizer wczytujemy z katalogu z adapterami.
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if not tokenizer.chat_template:
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3.5")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Wczytano model i adaptery z: {ADAPTER_DIR}")

## Formatowanie wejścia i funkcja predykcji

Używamy dokładnie tego samego promptu i formatu wejścia co podczas treningu.


In [ ]:
CLASSIFICATION_PROMPT = """Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
{verse}"""


def format_input(verse: str) -> str:
    messages = [{"role": "user", "content": CLASSIFICATION_PROMPT.format(verse=verse)}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        chat_template_kwargs={"enable_thinking": False},
    )


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def predict_artist(verse: str, top_k: int = 5):
    inputs = tokenizer(
        format_input(verse),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)
    probs = model(**inputs).logits.softmax(-1)[0]
    top = probs.topk(min(top_k, NUM_LABELS))
    return [(id2label[i.item()], p.item()) for p, i in zip(top.values, top.indices)]

## Inferencja dla wybranej zwrotki ze zbioru testowego


In [ ]:
SAMPLE_INDEX = 0  # Indeks zwrotki ze zbioru testowego

row = test_df.iloc[SAMPLE_INDEX]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")

## Inferencja dla losowej zwrotki ze zbioru testowego


In [ ]:
row = test_df.sample(1).iloc[0]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")